In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import math

# Data processing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

# Train model 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.model_selection import GridSearchCV, cross_val_score

from imblearn.over_sampling import SMOTE

# Warnings
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

In [7]:
# Load data 
df = pd.read_csv("heart_disease_uci.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [11]:
# Xu ly nhan + drop 
df['num'] = df['num'].apply(lambda x: 1 if x > 0 else 0)
df = df.drop(columns=['id', 'dataset'], errors='ignore')

# Thống kê mô tả
eda_summary = df.describe().T
print(eda_summary)
eda_summary.to_csv('eda_summary_statistics.csv')

# Tạo thư mục lưu hình EDA
if not os.path.exists("eda_plots"):
    os.makedirs("eda_plots")

# Vẽ phân bố Target
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='num', palette='Set2')
plt.title("Phân bố biến mục tiêu (Target Distribution)")
plt.savefig("eda_plots/1_target_distribution.png")
plt.close()

# Tách riêng cột số để vẽ đồ thị
numeric_cols_eda = df.select_dtypes(include=[np.number]).columns.drop('num', errors='ignore')
n_cols = len(numeric_cols_eda)
cols_plot = 4
rows_plot = math.ceil(n_cols / cols_plot)

# Kiểm tra Outlier
plt.figure(figsize=(15, rows_plot * 3))
for i, col in enumerate(numeric_cols_eda, 1):
    plt.subplot(rows_plot, cols_plot, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f"Boxplot: {col}")
    plt.tight_layout()
plt.savefig("eda_plots/2_numeric_boxplots.png")
plt.close()

# Histogram kiểm tra phân phối (Skewness)
plt.figure(figsize=(15, rows_plot * 3))
for i, col in enumerate(numeric_cols_eda, 1):
    plt.subplot(rows_plot, cols_plot, i)
    sns.histplot(df[col], kde=True, color='salmon')
    plt.title(f"Histogram: {col}")
    plt.tight_layout()
plt.savefig("eda_plots/3_numeric_histograms.png")
plt.close()

print("\nĐã xuất các bảng thống kê và biểu đồ EDA vào thư mục 'eda_plots/'!")


          count        mean         std   min    25%    50%    75%    max
age       920.0   53.510870    9.424685  28.0   47.0   54.0   60.0   77.0
trestbps  861.0  132.132404   19.066070   0.0  120.0  130.0  140.0  200.0
chol      890.0  199.130337  110.780810   0.0  175.0  223.0  268.0  603.0
thalch    865.0  137.545665   25.926276  60.0  120.0  140.0  157.0  202.0
oldpeak   858.0    0.878788    1.091226  -2.6    0.0    0.5    1.5    6.2
ca        309.0    0.676375    0.935653   0.0    0.0    0.0    1.0    3.0
num       920.0    0.553261    0.497426   0.0    0.0    1.0    1.0    1.0

Đã xuất các bảng thống kê và biểu đồ EDA vào thư mục 'eda_plots/'!


In [17]:
# KIỂM TRA MISSING  
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing': missing,
    'Percent': missing_percent
})

print(missing_df[missing_df['Missing'] > 0])

          Missing    Percent
trestbps       59   6.413043
chol           30   3.260870
fbs            90   9.782609
restecg         2   0.217391
thalch         55   5.978261
exang          55   5.978261
oldpeak        62   6.739130
slope         309  33.586957
ca            611  66.413043
thal          486  52.826087


In [18]:
# LỌC TƯƠNG QUAN 
numeric_cols = df.drop(columns=['num']).select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr().abs()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='RdBu_r')
plt.title("Correlation Heatmap")
plt.savefig("eda_plots/4_correlation_heatmap.png")
plt.close()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.9)]

df = df.drop(columns=to_drop)

print("\nCORRELATION FILTER")
print("Drop:", to_drop if len(to_drop) > 0 else "Không có")


CORRELATION FILTER
Drop: Không có


In [12]:
 
X = df.drop(columns=['num'])
y = df['num']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTRAIN TEST SPLIT")
print("Train:", X_train.shape)
print("Test:", X_test.shape)
 
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()



TRAIN TEST SPLIT
Train: (736, 13)
Test: (184, 13)


In [13]:
# Pipeline
numeric_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', RobustScaler()),
    ('variance', VarianceThreshold(0.01))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Transform data
preprocessor.fit(X_train)

X_train_pre = preprocessor.transform(X_train)
X_test_pre = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_pre, y_train)

In [14]:
# Feature Selection
k = min(15, len(feature_names))

selector = SelectKBest(score_func=mutual_info_classif, k=k)
np.random.seed(42)
X_train_final = selector.fit_transform(X_train_bal, y_train_bal)
X_test_final = selector.transform(X_test_pre)

mask = selector.get_support()
selected_features = feature_names[mask]

feature_score_df = pd.DataFrame({
    'Feature': feature_names,
    'Score': selector.scores_
}).sort_values(by='Score', ascending=False)
 
print(feature_score_df.head(10))
 
df_train_cleaned = pd.DataFrame(X_train_final, columns=selected_features)
df_train_cleaned['target'] = y_train_bal.values

df_test_cleaned = pd.DataFrame(X_test_final, columns=selected_features)
df_test_cleaned['target'] = y_test.values


                        Feature     Score
13              cat__exang_True  0.129176
5                       num__ca  0.120972
3                   num__thalch  0.111058
4                  num__oldpeak  0.088081
0                      num__age  0.082103
2                     num__chol  0.081890
16             cat__thal_normal  0.064657
6                 cat__sex_Male  0.058801
17  cat__thal_reversable defect  0.043003
15         cat__slope_upsloping  0.041032


In [16]:
# So sánh nhãn Target trước và sau SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x=y_train, ax=axes[0], palette='Set2')
axes[0].set_title('Mất cân bằng Target (Trước SMOTE)')
sns.countplot(x=y_train_bal, ax=axes[1], palette='Set2')
axes[1].set_title('Cân bằng Target (Sau SMOTE)')
plt.tight_layout()
plt.savefig('eda_plots/5_target_before_after_smote.png')
plt.close()

# Vẽ biểu đồ không gian PCA so sánh cấu trúc dữ liệu
pca = PCA(n_components=2, random_state=42)
# Khớp (Fit) PCA trên dữ liệu TRƯỚC SMOTE
X_pca_before = pca.fit_transform(X_train_pre)
# Biến đổi (Transform) dữ liệu SAU SMOTE dựa trên cùng hệ trục tọa độ
X_pca_after = pca.transform(X_train_bal)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Đồ thị trước SMOTE
scatter_before = axes[0].scatter(X_pca_before[:, 0], X_pca_before[:, 1], c=y_train,
                                 cmap='coolwarm', alpha=0.6, edgecolors='k')
axes[0].set_title('Không gian Dữ liệu TRƯỚC SMOTE (PCA 2D)', fontsize=14)
axes[0].set_xlabel('Thành phần chính 1')
axes[0].set_ylabel('Thành phần chính 2')

# Đồ thị sau SMOTE
scatter_after = axes[1].scatter(X_pca_after[:, 0], X_pca_after[:, 1], c=y_train_bal,
                                cmap='coolwarm', alpha=0.6, edgecolors='k')
axes[1].set_title('Không gian Dữ liệu SAU SMOTE (PCA 2D)', fontsize=14)
axes[1].set_xlabel('Thành phần chính 1')

handles, labels = scatter_before.legend_elements()
fig.legend(handles, ['Không bệnh (0)', 'Có bệnh (1)'], loc='upper center', ncol=2, fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('eda_plots/6_pca_space_comparison.png')
plt.close()
 
df_train_cleaned.to_csv('heart_train_cleaned.csv', index=False)
df_test_cleaned.to_csv('heart_test_cleaned.csv', index=False)

joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(selector, 'selector.pkl')
 
print("Train:", df_train_cleaned.shape)
print("Test:", df_test_cleaned.shape)
print("Số features giữ lại:", len(selected_features))

Train: (814, 16)
Test: (184, 16)
Số features giữ lại: 15


In [19]:
# LOAD PREPROCESSED DATA
df_train = pd.read_csv('heart_train_cleaned.csv')
df_test  = pd.read_csv('heart_test_cleaned.csv')

X_train = df_train.drop(columns=['target'])
y_train = df_train['target']

X_test = df_test.drop(columns=['target'])
y_test = df_test['target']

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# HYPERPARAMETER TUNING
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train) # Train model

best_params = grid_search.best_params_
rf_model    = grid_search.best_estimator_

print(f"Bộ tham số tốt nhất: {best_params}")
print(f"Model tốt nhất: {rf_model}")

joblib.dump(rf_model, 'rf_model.pkl') # Save file

# PREDICT & EVALUATE
y_pred       = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

accuracy    = accuracy_score(y_test, y_pred)
auc         = roc_auc_score(y_test, y_pred_proba)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print(f"\n{'='*45}")
print("KẾT QUẢ ĐÁNH GIÁ")
print(f"{'='*45}")
print(f"Accuracy:             {round(accuracy * 100, 2)}%")
print(f"ROC-AUC:              {round(auc, 4)}")
print(f"Sensitivity (Recall): {round(sensitivity, 4)}")
print(f"Specificity:          {round(specificity, 4)}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Không bệnh', 'Có bệnh']))

cv_scores = cross_val_score(rf_model, X_train, y_train, cv=10, scoring='accuracy')

print(f"\n{'='*45}")
print("K-FOLD CROSS-VALIDATION (K=10)")
print(f"{'='*45}")
print(f"Các mức Accuracy từng Fold: {cv_scores}")
print(f"Accuracy trung bình (Mean): {round(cv_scores.mean() * 100, 2)}%")
print(f"Độ lệch chuẩn (Std Dev):    {round(cv_scores.std() * 100, 2)}%")

 

Train: (814, 15) | Test: (184, 15)
Bộ tham số tốt nhất: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Model tốt nhất: RandomForestClassifier(max_depth=20, n_estimators=300, random_state=42)

KẾT QUẢ ĐÁNH GIÁ
Accuracy:             87.5%
ROC-AUC:              0.9189
Sensitivity (Recall): 0.9216
Specificity:          0.8171

Classification Report:
              precision    recall  f1-score   support

  Không bệnh       0.89      0.82      0.85        82
     Có bệnh       0.86      0.92      0.89       102

    accuracy                           0.88       184
   macro avg       0.88      0.87      0.87       184
weighted avg       0.88      0.88      0.87       184


K-FOLD CROSS-VALIDATION (K=10)
Các mức Accuracy từng Fold: [0.84146341 0.80487805 0.84146341 0.84146341 0.79012346 0.79012346
 0.81481481 0.80246914 0.91358025 0.90123457]
Accuracy trung bình (Mean): 83.42%
Độ lệch chuẩn (Std Dev):    4.13%


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix 
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Không bệnh', 'Có bệnh'],
            yticklabels=['Không bệnh', 'Có bệnh'])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Thực tế')
axes[0].set_xlabel('Dự đoán')

# ROC Curve  
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='blue', lw=2,
             label=f'AUC = {round(auc, 4)}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()